In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random

# ---- Config (paper defaults) ----
D_MODEL    = 1024   # LLM hidden dim; override per model (Qwen3-0.6B=1024, 1.7B=2048)
D_BOT      = 256    # VQ-VAE bottleneck dim (encoder output / codebook dim)
L          = 16     # compression rate: L text tokens → 1 latent token
C_SIZE     = 1024   # codebook size
BETA       = 0.25   # commitment loss weight

LR_VQVAE  = 1e-5   # paper: Adam lr=1e-5
BATCH_SIZE = 32     # paper: batch=32
STEPS      = 500    # demo steps (paper: 100k)

# Randomized replacement schedule (paper: M = multiples of L up to 256)
M_SET = [0, 72, 128, 160, 192, 224, 256]

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
print(f"L={L}, C_SIZE={C_SIZE}, D_model={D_MODEL}, D_bot={D_BOT}")

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


L=16, C_SIZE=1024, D_model=1024, D_bot=256


In [2]:
import sys
sys.path.insert(0, "/Users/fangyuanyu/Implementation/mod_gpt")

from sorl.tokenassort import (
    TokenAssortedVQVAE,
    sample_replacement_length,
    assign_latent_ids,
    build_mixed_sequence,
    add_abs_special_tokens,
    DEFAULT_L, DEFAULT_C_SIZE, DEFAULT_D_BOT, DEFAULT_BETA, DEFAULT_M_SET,
)

print("Imported from sorl.tokenassort")

Imported from sorl.tokenassort


In [3]:
# Baseline to compare against: TokenAssorted
# [26]. Token Assorted: Mixing Latent and Text Tokens for Improved Language Model Reasoning

# VQ-VAE Training For each benchmark, we train a VQVAE for 100k steps using the Adam optimizer, with learning rate  10−5 and batch size 32. We use a codebook of size 1024 and compress every chunk of L = 16 tokens into a single latent token (i.e., the compression rate r = 16).

# Randomized Latent Code Replacement We introduce a stochastic procedure for partially replacing CoT tokens with latent codes. Specifically, we define a set of predetermined numbers M = {0, 72, 128, 160, 192, 224, 256}, which are all multipliers of L = 16. For each training example, we first sample  mmax ∈ Mthen sample an integer m ∈ [0, 16, 32, . . . , mmax]uniformly at random. The first m CoT tokens are replaced by their corresponding latent discrete codes, while the later ones remain as raw text. This stochastic replacement mechanism exposes the model to a wide range of latent-text mixtures, enabling it to effectively learn from varying degrees of latent abstraction.
# Note: they put abstract tokens inside </abs_begin> ... </abs_end> braket, so the data processing function needs to be updated accordingly
#       I assume during evaluation time, they directly use the model to generate CoT with abstraction involved

In [3]:
# ---- (3) Full GSM8K pipeline ----
# Step A: load dataset, parse question / CoT / answer
# Step B: extract CoT embeddings from a frozen LLM
# Step C: train VQ-VAE on the extracted embeddings (→ labeler)
# Step D: for each training sample, call build_mixed_sequence → mixed_ids

from datasets import load_dataset

# --- A. Load GSM8K ---
gsm = load_dataset("gsm8k", "main")
train_ds = gsm["train"]   # 7473 samples; each has "question" and "answer"

def parse_gsm8k(sample):
    """Split GSM8K answer into CoT reasoning and final answer (after '####')."""
    answer_str = sample["answer"]
    if "####" in answer_str:
        cot_text, final = answer_str.split("####", 1)
        return sample["question"].strip(), cot_text.strip(), final.strip()
    return sample["question"].strip(), answer_str.strip(), ""

In [4]:
# ---- Step B: Extract CoT embeddings from a frozen LLM ----
# Replace the mock below with your actual model + tokenizer.
# The labeler only needs token embeddings (layer 0 = token embedding table),
# or any intermediate hidden state — layer 0 is cheapest and avoids loading
# the full model on a CPU notebook.

from transformers import AutoTokenizer, AutoModel
import torch

MODEL_ID = "Qwen/Qwen3-0.6B"

print(f"Loading tokenizer for {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# --- extract token embedding matrix (no forward pass needed) ---
print(f"Loading embedding table only (no full forward pass) ...")
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True)
emb_table = model.get_input_embeddings().weight.detach().float()  # cast bf16 → float32
D_MODEL_REAL = emb_table.shape[1]
print(f"Embedding table: {emb_table.shape}   dtype={emb_table.dtype}   D_model={D_MODEL_REAL}")

# Update config to match real D_MODEL
D_MODEL = D_MODEL_REAL

Loading tokenizer for Qwen/Qwen3-0.6B ...
Loading embedding table only (no full forward pass) ...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3Model LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding table: torch.Size([151936, 1024])   dtype=torch.float32   D_model=1024


In [5]:
# ---- Step C prep: extract chunk token IDs from full text ----
# Store (n_chunks, L) int64 IDs instead of embeddings — only ~10 MB for all GSM8K.
# Embeddings are looked up on-the-fly during VQ-VAE training: emb_table[batch_ids].

def get_full_text_chunk_ids(sample, tokenizer, L=16):
    """FULL text (q + cot + ans) → (n_chunks, L) token-ID tensor."""
    full_text = sample["question"] + " " + sample["answer"]
    ids = tokenizer(full_text, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    n_chunks = len(ids) // L
    if n_chunks == 0:
        return None
    return ids[:n_chunks * L].reshape(n_chunks, L)   # (n_chunks, L)

N_SAMPLES_EMB = len(train_ds)
print(f"Extracting chunk token IDs from {N_SAMPLES_EMB} samples ...")
chunk_id_list = []
for i in range(N_SAMPLES_EMB):
    cids = get_full_text_chunk_ids(train_ds[i], tokenizer, L)
    if cids is not None:
        chunk_id_list.append(cids)

all_chunk_ids = torch.cat(chunk_id_list, dim=0)   # (N_total, L)  int64
print(f"Total chunks : {all_chunk_ids.shape[0]}  shape: {all_chunk_ids.shape}")
print(f"Memory (IDs) : {all_chunk_ids.nbytes / 1024**2:.1f} MB  (vs {all_chunk_ids.shape[0] * L * D_MODEL * 4 / 1024**2:.0f} MB if stored as float32)")

Extracting chunk token IDs from 7473 samples ...
Total chunks : 81221  shape: torch.Size([81221, 16])
Memory (IDs) : 9.9 MB  (vs 5076 MB if stored as float32)


In [6]:
# ---- Step C: Train TokenAssortedVQVAE (EMA codebook + dead-code reinit) ----
# Encoder: (B, L, D) → flatten → nn.Linear(L*D, D_bot) → (B, D_bot)
# VQ:      EMA codebook update + dead-code reinitialization (standard VQ-VAE-2 trick)
# Decoder: (B, D_bot) → nn.Linear(D_bot, L*D) → (B, L, D)
#
# Codebook collapse fix:
#   - Codebook entries move via EMA toward their assigned encoder outputs (no gradient).
#   - Entries with EMA usage < dead_threshold are reborn from random batch elements.
#   - Only the encoder sees gradients (commitment loss: beta * ||sg(e_k) - z||^2).
#
# Monitoring (from MishaLaskin/vqvae reference):
#   perplexity = exp(entropy of batch assignment dist); max = C_SIZE (fully uniform).

from sorl.tokenassort import TokenAssortedVQVAE

torch.manual_seed(SEED)
vqvae = TokenAssortedVQVAE(D_MODEL, L=L, D_bot=D_BOT, C_SIZE=C_SIZE, beta=BETA,
                            decay=0.99, dead_threshold=0.01)
# Only encoder + decoder parameters need optimizer — codebook updated via EMA.
enc_dec_params = list(vqvae.encoder.parameters()) + list(vqvae.decoder.parameters())
opt = torch.optim.Adam(enc_dec_params, lr=LR_VQVAE)

STEPS_VQVAE = 20000
print(f"Training TokenAssortedVQVAE (EMA codebook, dead_threshold=0.01) for {STEPS_VQVAE} steps ...")
print(f"  encoder: ({L}×{D_MODEL}) → D_bot={D_BOT} → codebook {C_SIZE}  (max perplexity={C_SIZE})")
print(f"{'step':>6}  {'recon':>8}  {'commit':>8}  {'total':>8}  {'perplexity':>12}  {'vocab_util':>10}")
print("-" * 68)

for step in range(STEPS_VQVAE):
    idx = torch.randperm(len(all_chunk_ids))[:BATCH_SIZE]
    x_b = emb_table[all_chunk_ids[idx]]           # (B, L, D)  — on-the-fly lookup
    ids_out, x_hat, recon_loss, commit_loss, total_loss = vqvae(x_b)
    opt.zero_grad(); total_loss.backward(); opt.step()

    if (step + 1) % 200 == 0:
        with torch.no_grad():
            s_idx = torch.randperm(len(all_chunk_ids))[:2048]
            x_s   = emb_table[all_chunk_ids[s_idx]]
            
            # Vocab util over larger sample
            util = vqvae.vocab_utilization(x_s)
            
            # Perplexity over training batch
            e_mean = F.one_hot(ids_out, C_SIZE).float().mean(0)           # (C,)
            perplexity = torch.exp(-(e_mean * (e_mean + 1e-10).log()).sum()).item()
            
        print(f"{step+1:>6}  {recon_loss.item():>8.4f}  {commit_loss.item():>8.4f}  {total_loss.item():>8.4f}  {perplexity:>12.1f}  {util:>10.3f}")

print("\nvqvae is now the chunk labeler for GSM8K.")

Training TokenAssortedVQVAE (EMA codebook, dead_threshold=0.01) for 20000 steps ...
  encoder: (16×1024) → D_bot=256 → codebook 1024  (max perplexity=1024)
  step     recon    commit     total    perplexity  vocab_util
--------------------------------------------------------------------
   200    0.0019    0.0000    0.0019           3.2       0.018
   400    0.0017    0.0000    0.0017           4.4       0.014
   600    0.0014    0.0000    0.0014           4.0       0.016
   800    0.0010    0.0000    0.0010           4.9       0.017
  1000    0.0008    0.0000    0.0009           6.1       0.019
  1200    0.0007    0.0000    0.0008          10.4       0.029
  1400    0.0007    0.0000    0.0007           8.6       0.032
  1600    0.0007    0.0000    0.0007          14.8       0.035
  1800    0.0006    0.0000    0.0006          14.4       0.037
  2000    0.0007    0.0000    0.0007          14.4       0.035
  2200    0.0006    0.0000    0.0007          15.0       0.042
  2400    0.0007   

In [8]:
# ---- Step D: Build mixed sequences using trained vqvae ----

from sorl.tokenassort import add_abs_special_tokens, sample_replacement_length

abs_begin_id, abs_end_id = add_abs_special_tokens(tokenizer)
LATENT_OFFSET = len(tokenizer)

def get_cot_chunks(cot_text, tokenizer, emb_table, L=16):
    """CoT text → (n_chunks, L, D_MODEL) raw chunk tensor for labeling."""
    ids = tokenizer(cot_text, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    n_chunks = len(ids) // L
    if n_chunks == 0:
        return torch.zeros(0, L, emb_table.shape[1])
    ids_trunc = ids[:n_chunks * L].reshape(n_chunks, L)
    return emb_table[ids_trunc]   # (n_chunks, L, D)

def prepare_sample(sample, tokenizer, emb_table, vqvae, latent_offset,
                   abs_begin_id=None, abs_end_id=None, L=16, M_set=None):
    """Full pipeline: raw (n_chunks, L, D) chunks → vqvae.encode() → mixed sequence."""
    q_text, cot_text, ans_text = parse_gsm8k(sample)
    q_ids   = tokenizer(q_text,   add_special_tokens=False)["input_ids"]
    cot_ids = tokenizer(cot_text, add_special_tokens=False)["input_ids"]
    ans_ids = tokenizer(ans_text, add_special_tokens=False)["input_ids"]

    cot_chunks = get_cot_chunks(cot_text, tokenizer, emb_table, L)
    n_chunks   = cot_chunks.shape[0]

    m         = sample_replacement_length(len(cot_ids), M_set=M_set, L=L)
    n_replace = m // L

    mixed = list(q_ids)
    if n_replace > 0 and n_chunks > 0:
        if abs_begin_id is not None:
            mixed.append(abs_begin_id)
        with torch.no_grad():
            lat_ids = vqvae.encode(cot_chunks)   # (n_chunks,)
        for k in range(min(n_replace, n_chunks)):
            mixed.append(int(lat_ids[k].item()) + latent_offset)
        if abs_end_id is not None:
            mixed.append(abs_end_id)

    mixed += list(cot_ids[m:])
    if ans_ids:
        mixed += list(ans_ids)
    return mixed, m

print(f"abs_begin_id  : {abs_begin_id}  ({tokenizer.convert_ids_to_tokens(abs_begin_id)})")
print(f"abs_end_id    : {abs_end_id}  ({tokenizer.convert_ids_to_tokens(abs_end_id)})")
print(f"Latent range  : [{LATENT_OFFSET}, {LATENT_OFFSET + C_SIZE})")
print(f"\n{'idx':>4}  {'m':>4}  {'n_lat':>6}  {'n_txt_cot':>9}  {'total_len':>9}")
print("-" * 42)
for i in range(8):
    mixed, m = prepare_sample(train_ds[i], tokenizer, emb_table, vqvae,
                               LATENT_OFFSET, abs_begin_id, abs_end_id)
    _, cot_i, _ = parse_gsm8k(train_ds[i])
    T_cot = len(tokenizer(cot_i, add_special_tokens=False)["input_ids"])
    print(f"{i:>4}  {m:>4}  {m//L:>6}  {T_cot - m:>9}  {len(mixed):>9}")

abs_begin_id  : 151669  (<abs_begin>)
abs_end_id    : 151670  (<abs_end>)
Latent range  : [151671, 152695)

 idx     m   n_lat  n_txt_cot  total_len
------------------------------------------
   0     0       0         55         96
   1     0       0         60         93
   2    16       1         81        146
   3     0       0        122        178
   4    64       4          7         42
   5     0       0        158        225
   6     0       0         83        142
   7     0       0        110        218


In [13]:
mixed, m = prepare_sample(train_ds[4], tokenizer, emb_table, vqvae,
                           LATENT_OFFSET, abs_begin_id, abs_end_id)

mixed

[28084,
 13914,
 264,
 220,
 18,
 15112,
 6524,
 311,
 220,
 17,
 2155,
 4780,
 10917,
 264,
 2003,
 13,
 220,
 2585,
 1657,
 6816,
 1558,
 566,
 3270,
 264,
 1042,
 30,
 1519,
 13914,
 1817,
 4238,
 220,
 18,
 9,
 17,
 28,
 2442,
 18,
 9,
 17,
 28,
 21,
 2452,
 21,
 6816,
 264,
 2003,
 198,
 4416,
 566,
 13914,
 220,
 21,
 9,
 17,
 28,
 2442,
 21,
 9,
 17,
 28,
 16,
 17,
 2452,
 16,
 17,
 6816,
 1449,
 2003,
 198,
 4792,
 3363,
 566,
 13914,
 220,
 16,
 17,
 9,
 20,
 17,
 28,
 2442,
 16,
 17,
 9,
 20,
 17,
 28,
 21,
 17,
 19,
 2452,
 21,
 17,
 19,
 6816,
 264,
 1042,
 21,
 17,
 19]

In [14]:
# ---- Inspection: decode a mixed sequence back to readable form ----
# Shows exactly what the model sees: which positions are latent codes vs text tokens.

def inspect_mixed_sequence(mixed_ids, tokenizer, latent_offset, abs_begin_id, abs_end_id):
    """
    Pretty-print a mixed sequence, labelling each token as:
      [TEXT]   → normal text token (decoded)
      [ABS_BEGIN] / [ABS_END]  → bracket special tokens
      [LAT:k]  → latent code k (abstract token)
    """
    lines = []
    for tid in mixed_ids:
        if tid == abs_begin_id:
            lines.append("<abs_begin>")
        elif tid == abs_end_id:
            lines.append("<abs_end>")
        elif tid >= latent_offset:
            lines.append(f"[LAT:{tid - latent_offset}]")
        else:
            tok = tokenizer.decode([tid], skip_special_tokens=False)
            lines.append(repr(tok))
    return " ".join(lines)


# Inspect sample 1 — run a few times (stochastic m) to see different mix ratios
random.seed(0)
for trial in range(3):
    mixed, m = prepare_sample(train_ds[1], tokenizer, emb_table, vqvae,
                               LATENT_OFFSET, abs_begin_id, abs_end_id)
    _, cot_i, _ = parse_gsm8k(train_ds[1])
    n_lat = m // L
    print(f"\n--- trial {trial}  m={m}  ({n_lat} latent tokens replacing {m} text tokens) ---")
    readable = inspect_mixed_sequence(mixed, tokenizer, LATENT_OFFSET, abs_begin_id, abs_end_id)
    print(readable[:600], "..." if len(readable) > 600 else "")


--- trial 0  m=48  (3 latent tokens replacing 48 text tokens) ---
'W' 'eng' ' earns' ' $' '1' '2' ' an' ' hour' ' for' ' babys' 'itting' '.' ' Yesterday' ',' ' she' ' just' ' did' ' ' '5' '0' ' minutes' ' of' ' babys' 'itting' '.' ' How' ' much' ' did' ' she' ' earn' '?' <abs_begin> [LAT:170] [LAT:139] [LAT:137] <abs_end> '.' '2' '*' '5' '0' '=' '1' '0' '>>' '1' '0' '.' '1' '0' 

--- trial 1  m=48  (3 latent tokens replacing 48 text tokens) ---
'W' 'eng' ' earns' ' $' '1' '2' ' an' ' hour' ' for' ' babys' 'itting' '.' ' Yesterday' ',' ' she' ' just' ' did' ' ' '5' '0' ' minutes' ' of' ' babys' 'itting' '.' ' How' ' much' ' did' ' she' ' earn' '?' <abs_begin> [LAT:170] [LAT:139] [LAT:137] <abs_end> '.' '2' '*' '5' '0' '=' '1' '0' '>>' '1' '0' '.' '1' '0' 

--- trial 2  m=0  (0 latent tokens replacing 0 text tokens) ---
'W' 'eng' ' earns' ' $' '1' '2' ' an' ' hour' ' for' ' babys' 'itting' '.' ' Yesterday' ',' ' she' ' just' ' did' ' ' '5' '0' ' minutes' ' of' ' babys' 'itting' '.' ' Ho

In [15]:
# ---- Step E: Create PyTorch Dataset & DataLoader for SFT ----
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class MixedSequenceDataset(Dataset):
    """
    On-the-fly generation of mixed sequences using the frozen VQ-VAE.
    Produces input_ids, attention_mask, labels (for standard causal LM training).
    """
    def __init__(self, data, tokenizer, emb_table, vqvae, L=16):
        self.data      = data
        self.tokenizer = tokenizer
        self.emb_table = emb_table
        self.vqvae     = vqvae
        self.L         = L
        self.latent_offset = len(tokenizer) - C_SIZE - 2

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Generate mixed sequence using VQ-VAE
        mixed_ids, m = prepare_sample(
            self.data[idx], self.tokenizer, self.emb_table, self.vqvae,
            self.latent_offset, abs_begin_id, abs_end_id, L=self.L
        )
        
        # Parse text to find where the question ends (prompt_len)
        q_text, _, _ = parse_gsm8k(self.data[idx])
        q_ids = self.tokenizer(q_text, add_special_tokens=False)["input_ids"]
        prompt_len = len(q_ids)

        return {
            "input_ids": torch.tensor(mixed_ids, dtype=torch.long),
            "prompt_len": prompt_len
        }

def collate_fn(batch, pad_token_id):
    """Pads sequences and builds standard causal LM labels (-100 for prompt & padding)."""
    input_ids = [item["input_ids"] for item in batch]
    prompt_lens = [item["prompt_len"] for item in batch]

    # Pad sequences
    padded_input_ids = pad_sequence(input_ids, batch_first=True, padding_value=pad_token_id)
    attention_mask = (padded_input_ids != pad_token_id).long()

    # Labels: shift input_ids, ignore padding and prompt
    labels = padded_input_ids.clone()
    labels[labels == pad_token_id] = -100
    
    # Mask out the prompt (question) so loss is only computed on CoT + Answer
    for i, p_len in enumerate(prompt_lens):
        labels[i, :p_len] = -100

    return {
        "input_ids": padded_input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

# Create Dataset & DataLoader
mixed_train_ds = MixedSequenceDataset(train_ds, tokenizer, emb_table, vqvae, L=L)
pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

# Use a small batch size for demonstration
train_loader = DataLoader(
    mixed_train_ds,
    batch_size=2,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, pad_id)
)

# Inspect a single batch
batch = next(iter(train_loader))
print(f"input_ids shape      : {batch['input_ids'].shape}")
print(f"attention_mask shape : {batch['attention_mask'].shape}")
print(f"labels shape         : {batch['labels'].shape}")

# Verify labels mask the prompt (-100)
print(f"\nExample row 0 labels (first 40 tokens, -100 means ignored):")
print(batch['labels'][0, :40].tolist())

input_ids shape      : torch.Size([2, 180])
attention_mask shape : torch.Size([2, 180])
labels shape         : torch.Size([2, 180])

Example row 0 labels (first 40 tokens, -100 means ignored):
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


In [16]:
# ---- Step F: Minimal SFT Training Loop with Qwen ----
# Since Qwen3 needs to embed the new latent tokens, we must resize its embeddings
# AND ensure we don't destroy the original pre-trained embeddings in the process.

from transformers import AutoModelForCausalLM

# 1. Load the actual model for training
print(f"Loading {MODEL_ID} for causal LM training...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 2. Resize embeddings to accommodate the extra tokens (abs_begin, abs_end, +1024 latents)
old_vocab_size = model.config.vocab_size
new_vocab_size = len(tokenizer)  # Should be 151643 + 2 + 1024 = 152669

print(f"\nResizing model embeddings from {old_vocab_size} -> {new_vocab_size}")
model.resize_token_embeddings(new_vocab_size)

# 3. Initialize the new embedding vectors to the mean of existing vectors
# This prevents loss spikes at the start of training.
with torch.no_grad():
    emb = model.get_input_embeddings().weight
    mean_emb = emb[:old_vocab_size].mean(dim=0)
    emb[old_vocab_size:] = mean_emb

# 4. Minimal SFT loop
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
model.train()

print("\nStarting minimal SFT loop for 10 steps...")
for step, batch in enumerate(train_loader):
    if step >= 10:
        break
        
    input_ids = batch["input_ids"].to(model.device)
    attention_mask = batch["attention_mask"].to(model.device)
    labels = batch["labels"].to(model.device)
    
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )
    
    loss = outputs.loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Step {step+1:2d}/10 | Loss: {loss.item():.4f}")

print("\nSuccess! The mixed sequences are fully compatible with standard causal LM training.")

Loading Qwen/Qwen3-0.6B for causal LM training...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Resizing model embeddings from 151936 -> 151671

Starting minimal SFT loop for 10 steps...
Step  1/10 | Loss: 4.4803
Step  2/10 | Loss: 1.9531
Step  3/10 | Loss: 2.7169
Step  4/10 | Loss: 2.9767
Step  5/10 | Loss: 1.3342
Step  6/10 | Loss: 1.9172
Step  7/10 | Loss: 2.3034
Step  8/10 | Loss: 3.0192
Step  9/10 | Loss: 1.6165
Step 10/10 | Loss: 1.4972

Success! The mixed sequences are fully compatible with standard causal LM training.
